In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# 1. LOAD & INSPECT DATASET
# ==========================================
csv_file_path = "soil_fertility_classification_1000.csv"
df = pd.read_csv(csv_file_path)

print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df["Fertility_Class"].value_counts())

# ==========================================
# 2. FEATURE SELECTION & PREPROCESSING
# ==========================================
# Note: 'Fertility_Score' is excluded to avoid target leakage since 
# 'Fertility_Class' is directly partitioned from it (<40, 40-69.99, >=70).
# Identifier and metadata columns ('Sample_ID', 'Source_Calibration') are dropped.

feature_cols = [
    "pH", "Electrical_Conductivity_dS_m", "Organic_Carbon_g_kg",
    "Available_N_kg_ha", "Available_P_kg_ha", "Available_K_kg_ha",
    "Sulphur_mg_kg", "Zinc_mg_kg", "Boron_mg_kg", "Iron_mg_kg",
    "Manganese_mg_kg", "Copper_mg_kg"
]

target_col = "Fertility_Class"

X = df[feature_cols]
y = df[target_col]

# Stratified 80-20 train-test split to preserve class imbalance ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize features for distance- and margin-sensitive models (KNN, SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 3. MODEL INITIALIZATION & TRAINING
# ==========================================
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "SVM": SVC(kernel="rbf", C=1.0, random_state=42)
}

results = []

for name, model in models.items():
    # Use scaled data for KNN and SVM; unscaled for tree-based models
    if name in ["KNN", "SVM"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 2),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

# Benchmark comparison table
results_df = pd.DataFrame(results)
print("\n=== Model Benchmark Summary ===")
print(results_df.to_string(index=False))

# Detailed Classification Report for Best Model (Random Forest)
rf_model = models["Random Forest"]
rf_preds = rf_model.predict(X_test)
print("\n=== Random Forest Classification Report ===")
print(classification_report(y_test, rf_preds))

# Confusion Matrix for Random Forest
print("=== Random Forest Confusion Matrix ===")
labels = sorted(y.unique())
cm = confusion_matrix(y_test, rf_preds, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Actual {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
print(cm_df)

# ==========================================
# 4. FEATURE IMPORTANCE (RANDOM FOREST)
# ==========================================
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n=== Feature Importances (Random Forest) ===")
print(importance_df.to_string(index=False))

# ==========================================
# 5. SOIL FERTILITY PREDICTION FUNCTION
# ==========================================
def predict_soil_fertility(sample_dict: dict) -> str:
    """
    Predict fertility class for a new soil sample dictionary.
    Keys required: feature_cols
    """
    input_df = pd.DataFrame([sample_dict])[feature_cols]
    prediction = rf_model.predict(input_df)[0]
    probabilities = rf_model.predict_proba(input_df)[0]
    prob_summary = dict(zip(rf_model.classes_, np.round(probabilities, 3)))
    return f"Prediction: {prediction} | Probabilities: {prob_summary}"

# Example test sample
example_sample = {
    "pH": 7.20,
    "Electrical_Conductivity_dS_m": 0.35,
    "Organic_Carbon_g_kg": 14.50,
    "Available_N_kg_ha": 235.0,
    "Available_P_kg_ha": 45.0,
    "Available_K_kg_ha": 280.0,
    "Sulphur_mg_kg": 18.5,
    "Zinc_mg_kg": 1.20,
    "Boron_mg_kg": 0.85,
    "Iron_mg_kg": 24.0,
    "Manganese_mg_kg": 12.0,
    "Copper_mg_kg": 0.80
}

print("\n=== Example Sample Inference ===")
print(predict_soil_fertility(example_sample))

Dataset Shape: (1000, 20)

Class Distribution:
Fertility_Class
Medium Fertility    596
Low Fertility       319
High Fertility       85
Name: count, dtype: int64

=== Model Benchmark Summary ===
        Model  Accuracy (%)  Precision  Recall  F1-Score
Decision Tree          82.0     0.8144   0.820    0.8161
Random Forest          88.5     0.8982   0.885    0.8713
          KNN          76.0     0.7645   0.760    0.7499
          SVM          87.0     0.8750   0.870    0.8669

=== Random Forest Classification Report ===
                  precision    recall  f1-score   support

  High Fertility       1.00      0.29      0.45        17
   Low Fertility       0.96      0.86      0.91        64
Medium Fertility       0.85      0.98      0.91       119

        accuracy                           0.89       200
       macro avg       0.94      0.71      0.76       200
    weighted avg       0.90      0.89      0.87       200

=== Random Forest Confusion Matrix ===
                         Pre